In [1]:
import os
import sys
import pandas as pd
import numpy as np
import scipy
from cv_tools.utils import COCOEvalPlus
import os

import plot_utils as pu
from cv_tools import utils
import config as C

%load_ext autoreload
%autoreload 1
%aimport config
%aimport cv_tools.utils
%aimport plot_utils

/Users/s2785075/miniconda3/envs/cv_tools/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
N_FOLDS = 3
CCF = True
results_dir = os.path.join(C.RESULTS_DIR, 'ENHANCE_final_revisions')

# Domain-based splitting evaluation

In [7]:
res_dir = 'FINAL_GROUPS' # domain-based results
eval_type = 'Groups' # domain-based evaluation
results_df = []
for model in ['dt2', 'yolo8']:
    for dataset in C.MAIN_DATASETS:
        for augm in [None, 'NMS50'] + C.ALL_PPS + ['resized_MSRCR_15_80_250_fk', 'resized_MSRCR']:
            if augm == 'NMS50' and dataset != 'DeepFish':
                # comparison of NMS threshold 0.7 vs 0.5 was done only for DeepFish
                continue
            train_data = f'{dataset}.CV.{eval_type}{f".{augm}" if augm else ""}'
            fn = os.path.join(results_dir, 'pickles', f'{res_dir}.{model}.{dataset}.{eval_type}.{augm}.pckl')
            if not os.path.exists(fn):
                try:
                    results = utils.read_and_eval_results(
                        train_datas=train_data,
                        models=[model],
                        model_selects=['best'],
                        seeds=C.ALL_SEEDS[dataset],
                        splits=['test'],
                        results_dir=os.path.join(results_dir, res_dir),
                        n_folds=N_FOLDS,
                        concat=CCF,
                        verbose=False,
                    )
                    utils.clean_cached_results(results)
                    utils.pickle_dump(results, fn)
                except:
                    continue
            else:
                results = utils.pickle_load(fn)

            r = results[train_data][model]['best']['test']
            aps = [r[s].precision.mean() for s in r]
            print(f'{model} {train_data} {np.mean(aps):.6f} {np.std(aps):.3f} {[round(ap, 3) for ap in aps]}')
            results_df.append((model, dataset, augm, eval_type, np.mean(aps), np.std(aps), aps))
results_df = pd.DataFrame.from_records(results_df, columns=['model', 'dataset', 'pps', 'eval', 'mAP', 'std', 'mAPs'])
results_df.to_csv(os.path.join(results_dir, f'{res_dir}.csv'), index=None)
results_df

dt2 DeepFish.CV.Groups 0.468151 0.055 [0.419, 0.545, 0.44]
dt2 DeepFish.CV.Groups.NMS50 0.475525 0.046 [0.448, 0.54, 0.438]
dt2 DeepFish.CV.Groups.unsharp_mask 0.469915 0.061 [0.468, 0.545, 0.397]
dt2 DeepFish.CV.Groups.adjust_gamma_down 0.452319 0.030 [0.432, 0.494, 0.431]
dt2 DeepFish.CV.Groups.adjust_gamma_up 0.464464 0.025 [0.474, 0.489, 0.431]
dt2 DeepFish.CV.Groups.adjust_log 0.481267 0.047 [0.48, 0.54, 0.424]
dt2 DeepFish.CV.Groups.adjust_sigmoid 0.413345 0.042 [0.403, 0.469, 0.368]
dt2 DeepFish.CV.Groups.DCP 0.466290 0.040 [0.485, 0.503, 0.411]
dt2 DeepFish.CV.Groups.CAP 0.471537 0.012 [0.488, 0.459, 0.468]
dt2 DeepFish.CV.Groups.FUnIE-GAN 0.461266 0.037 [0.428, 0.513, 0.442]
dt2 DeepFish.CV.Groups.CLAHE 0.499712 0.048 [0.464, 0.568, 0.467]
dt2 DeepFish.CV.Groups.autocontrast 0.487967 0.072 [0.436, 0.59, 0.439]
dt2 DeepFish.CV.Groups.grey_world 0.505223 0.040 [0.521, 0.545, 0.45]
dt2 DeepFish.CV.Groups.ace 0.498423 0.058 [0.448, 0.58, 0.467]
dt2 DeepFish.CV.Groups.ARCR 0.500150

,model,dataset,pps,eval,mAP,std,mAPs
0,dt2,DeepFish,None,Groups,0.468151,0.054874,"[0.41930351518172104, 0.5447968724647027, 0.44..."
1,dt2,DeepFish,NMS50,Groups,0.475525,0.045539,"[0.44846496988276124, 0.5396655405530153, 0.43..."
2,dt2,DeepFish,unsharp_mask,Groups,0.469915,0.060674,"[0.4680758227075385, 0.5451282646687295, 0.396..."
3,dt2,DeepFish,adjust_gamma_down,Groups,0.452319,0.029787,"[0.43179192284221707, 0.4944398469998151, 0.43..."
4,dt2,DeepFish,adjust_gamma_up,Groups,0.464464,0.024720,"[0.47389456268877356, 0.4889011532668118, 0.43..."
...,...,...,...,...,...,...,...
105,yolo8,Jellytoring,ARCR,Groups,0.447428,0.010548,"[0.4610701852302508, 0.44583193559763806, 0.43..."
106,yolo8,Jellytoring,MLLE,Groups,0.436538,0.037544,"[0.4896312980548013, 0.41034850972737963, 0.40..."
107,yolo8,Jellytoring,resized_MSRCR_15_80_250_new,Groups,0.456090,0.035755,"[0.5044337307964225, 0.4190796213080974, 0.444..."
108,yolo8,Jellytoring,resized_MSRCR_15_80_250_fk,Groups,0.462523,0.018279,"[0.4863945929914695, 0.4591778691073969, 0.441..."


# Do not concatenate folds -- needed for estimating variability of traning folds

In [9]:
res_dir = 'FINAL_GROUPS' # domain-based results
eval_type = 'Groups' # domain-based evaluation
augm = None
for model in ['dt2', 'yolo8']:
    for dataset in C.MAIN_DATASETS:
        train_data = f'{dataset}.CV.{eval_type}{f".{augm}" if augm else ""}'
        fn = os.path.join(results_dir, 'pickles', f'{res_dir}.{model}.{dataset}.{eval_type}.{augm}.concatFalse.pckl')
        if not os.path.exists(fn):
            try:
                results = utils.read_and_eval_results(
                    train_datas=train_data,
                    models=[model],
                    model_selects=['best'],
                    seeds=C.ALL_SEEDS[dataset],
                    splits=['test'],
                    results_dir=os.path.join(results_dir, res_dir),
                    n_folds=N_FOLDS,
                    concat=False,
                    verbose=False,
                )
                utils.clean_cached_results(results)
                utils.pickle_dump(results, fn)
            except:
                continue
        else:
            results = utils.pickle_load(fn)

        r = results[train_data][model]['best']['test']
        aps = [np.mean([r[s][f].precision.mean() for f in range(len(r[s]))]) for s in r]
        print(f'{model} {train_data} {np.mean(aps):.3f} {np.std(aps):.3f} {[round(ap, 3) for ap in aps]}')

dt2 DeepFish.CV.Groups 0.494 0.045 [0.473, 0.556, 0.453]


/Users/s2785075/work/projects/jellies/src/cv_tools/utils.py:546: RuntimeWarning: invalid value encountered in long_scalars
  recall = np.asarray([TPs[k] / (TPs[k] + FNs[k]) for k in range(len(self._paramsEval.catIds))])


dt2 GLOW_low_visibility_estuaries.CV.Groups 0.483 0.039 [0.531, 0.437, 0.483]


/Users/s2785075/work/projects/jellies/src/cv_tools/utils.py:545: RuntimeWarning: invalid value encountered in long_scalars
  precision = np.asarray([TPs[k] / (TPs[k] + FPs[k]) for k in range(len(self._paramsEval.catIds))])


dt2 Jellytoring.CV.Groups 0.418 0.026 [0.454, 0.406, 0.395]
yolo8 DeepFish.CV.Groups 0.440 0.048 [0.41, 0.508, 0.401]
yolo8 GLOW_low_visibility_estuaries.CV.Groups 0.423 0.023 [0.456, 0.409, 0.404]
yolo8 Jellytoring.CV.Groups 0.384 0.027 [0.421, 0.359, 0.372]


# S-UODAC

In [8]:
res_dir = 'FINAL_S-UODAC' # S-UODAC results
eval_type = 'NoValRepro' # evaluate to reproduce S-UODAC evaluation faithfully
results_df = []
dataset = 'S-UODAC'
for model in ['dt2', 'yolo3']:
    for augm in [
        None,
        'resized_MSRCR',
        'resized_MSRCR_15_80_250_new',
    ]:
        tmp_train_data = f'{dataset}.{eval_type}{"E12withGamma" if model == "dt2" else "E100" if model == "yolo3" else ""}{f".{augm}" if augm else ""}'
        fn = os.path.join(results_dir, 'pickles', f'{res_dir}.{model}.{dataset}.{eval_type}.{augm}.pckl')
        train_data = f'{dataset}.{eval_type}{f".{augm}" if augm else ""}'
        if not os.path.exists(fn):
            if model == 'yolo3':
                results = utils.read_and_eval_yolo3(
                    train_datas=tmp_train_data,
                    seeds=C.ALL_SEEDS[dataset],
                    results_dir=os.path.join(results_dir, res_dir),
                    verbose=False,
                    maxDets=300,
                    conf_thr=0.02,
                    fix_zero_ann_ids=True
                )
            else:
                results = utils.read_and_eval_results(
                    train_datas=tmp_train_data,
                    models=[model],
                    model_selects=['last'],
                    seeds=C.ALL_SEEDS[dataset],
                    splits=['test'],
                    results_dir=os.path.join(results_dir, res_dir),
                    n_folds=None,
                    verbose=False,
                    maxDets=300,
                    conf_thr=0.05,
                    fix_zero_ann_ids=True
                )
            utils.clean_cached_results(results)
            results[train_data] = results[tmp_train_data]
            results.pop(tmp_train_data)
            utils.pickle_dump(results, fn)
        else:
            results = utils.pickle_load(fn)

        print(train_data, model)
        r = results[train_data][model]['last']['test']
        aps = [r[s].precision.mean() for s in r]
        species_aps = np.asarray([r[s].precision.mean(axis=0) for s in r]).mean(axis=0)
        print(f'{model} {train_data} {np.mean(aps):.3f} {np.std(aps):.3f} {[round(ap, 3) for ap in aps]}')
        results_df.append([model, dataset, augm, eval_type, np.mean(aps), np.std(aps), aps] + species_aps.tolist())
results_df = pd.DataFrame.from_records(results_df, columns=['model', 'dataset', 'pps', 'eval', 'mAP', 'std', 'mAPs'] + [f's{i}_AP' for i in range(len(species_aps))])
results_df.to_csv(os.path.join(results_dir, f'{res_dir}.csv'), index=None)
(results_df.set_index(['model', 'dataset', 'pps', 'eval'])[['s0_AP', 's1_AP', 's2_AP', 's3_AP', 'mAP']] * 100).round(2)

S-UODAC.NoValRepro dt2
dt2 S-UODAC.NoValRepro 0.537 0.015 [0.516, 0.551, 0.544]
S-UODAC.NoValRepro.resized_MSRCR dt2
dt2 S-UODAC.NoValRepro.resized_MSRCR 0.645 0.013 [0.628, 0.66, 0.647]
S-UODAC.NoValRepro.resized_MSRCR_15_80_250_new dt2
dt2 S-UODAC.NoValRepro.resized_MSRCR_15_80_250_new 0.631 0.011 [0.617, 0.643, 0.632]
S-UODAC.NoValRepro yolo3
yolo3 S-UODAC.NoValRepro 0.382 0.042 [0.41, 0.413, 0.322]
S-UODAC.NoValRepro.resized_MSRCR yolo3
yolo3 S-UODAC.NoValRepro.resized_MSRCR 0.617 0.004 [0.616, 0.623, 0.613]
S-UODAC.NoValRepro.resized_MSRCR_15_80_250_new yolo3
yolo3 S-UODAC.NoValRepro.resized_MSRCR_15_80_250_new 0.613 0.004 [0.609, 0.618, 0.612]


s0_AP  s1_AP  s2_AP  \
model dataset pps                         eval                              
dt2   S-UODAC NaN                         NoValRepro  76.57  41.42  50.31   
              resized_MSRCR               NoValRepro  80.35  63.40  56.22   
              resized_MSRCR_15_80_250_new NoValRepro  80.94  62.80  56.23   
yolo3 S-UODAC NaN                         NoValRepro  68.67  31.10  29.33   
              resized_MSRCR               NoValRepro  78.00  58.08  54.33   
              resized_MSRCR_15_80_250_new NoValRepro  78.10  61.09  53.46   

                                                      s3_AP    mAP  
model dataset pps                         eval                      
dt2   S-UODAC NaN                         NoValRepro  46.46  53.69  
              resized_MSRCR               NoValRepro  58.00  64.49  
              resized_MSRCR_15_80_250_new NoValRepro  52.33  63.07  
yolo3 S-UODAC NaN                         NoValRepro  23.67  38.19  
              resized_MSRCR               NoValRepro  56.47  61.72  
              resized_MSRCR_15_80_250_new NoValRepro  52.53  61.29

# Random-splitting evaluation (ignore domains)

In [6]:
res_dir = 'FINAL_NO_GROUPS' # random-splitting results
eval_type = 'NoGroups' # random-splittin evaluation
results_df = []
for model in ['dt2', 'yolo8']:
    for dataset in C.MAIN_DATASETS:
        for augm in [None] + C.ALL_PPS:
            train_data = f'{dataset}.CV.{eval_type}{f".{augm}" if augm else ""}'
            fn = os.path.join(results_dir, 'pickles', f'{res_dir}.{model}.{dataset}.{eval_type}.{augm}.pckl')
            if not os.path.exists(fn):
                try:
                    results = utils.read_and_eval_results(
                        train_datas=train_data,
                        models=[model],
                        model_selects=['best'],
                        seeds=C.ALL_SEEDS[dataset],
                        splits=['test'],
                        results_dir=os.path.join(results_dir, res_dir),
                        n_folds=N_FOLDS,
                        concat=CCF,
                        verbose=False,
                    )
                    utils.clean_cached_results(results)
                    utils.pickle_dump(results, fn)
                except:
                    continue
            else:
                results = utils.pickle_load(fn)

            r = results[train_data][model]['best']['test']
            aps = [r[s].precision.mean() for s in r]
            print(f'{model} {train_data} {np.mean(aps):.3f} {np.std(aps):.3f} {[round(ap, 3) for ap in aps]}')
            results_df.append((model, dataset, augm, eval_type, np.mean(aps), np.std(aps), aps))
results_df = pd.DataFrame.from_records(results_df, columns=['model', 'dataset', 'pps', 'eval', 'mAP', 'std', 'mAPs'])
results_df.to_csv(os.path.join(results_dir, f'{res_dir}.csv'), index=None)
results_df

dt2 DeepFish.CV.NoGroups 0.963 0.002 [0.965, 0.96, 0.963]
dt2 DeepFish.CV.NoGroups.unsharp_mask 0.961 0.001 [0.962, 0.962, 0.959]
dt2 DeepFish.CV.NoGroups.adjust_gamma_down 0.961 0.001 [0.959, 0.962, 0.96]
dt2 DeepFish.CV.NoGroups.adjust_gamma_up 0.963 0.002 [0.963, 0.961, 0.965]
dt2 DeepFish.CV.NoGroups.adjust_log 0.961 0.001 [0.959, 0.961, 0.963]
dt2 DeepFish.CV.NoGroups.adjust_sigmoid 0.960 0.001 [0.959, 0.961, 0.96]
dt2 DeepFish.CV.NoGroups.DCP 0.961 0.002 [0.961, 0.959, 0.963]
dt2 DeepFish.CV.NoGroups.CAP 0.959 0.001 [0.96, 0.957, 0.96]
dt2 DeepFish.CV.NoGroups.FUnIE-GAN 0.956 0.001 [0.957, 0.955, 0.957]
dt2 DeepFish.CV.NoGroups.CLAHE 0.961 0.001 [0.96, 0.961, 0.961]
dt2 DeepFish.CV.NoGroups.autocontrast 0.961 0.001 [0.961, 0.962, 0.961]
dt2 DeepFish.CV.NoGroups.grey_world 0.962 0.001 [0.963, 0.961, 0.962]
dt2 DeepFish.CV.NoGroups.ace 0.962 0.001 [0.961, 0.962, 0.962]
dt2 DeepFish.CV.NoGroups.ARCR 0.961 0.001 [0.961, 0.96, 0.963]
dt2 DeepFish.CV.NoGroups.MLLE 0.962 0.002 [0.964, 0

,model,dataset,pps,eval,mAP,std,mAPs
0,dt2,DeepFish,None,NoGroups,0.962876,0.002009,"[0.9653215037420165, 0.9603998665738995, 0.962..."
1,dt2,DeepFish,unsharp_mask,NoGroups,0.961071,0.001353,"[0.961863285694112, 0.9621842288086494, 0.9591..."
2,dt2,DeepFish,adjust_gamma_down,NoGroups,0.960511,0.000977,"[0.9593386865696515, 0.9617305819731189, 0.960..."
3,dt2,DeepFish,adjust_gamma_up,NoGroups,0.962980,0.001518,"[0.9629220187835795, 0.9611513831646608, 0.964..."
4,dt2,DeepFish,adjust_log,NoGroups,0.960937,0.001419,"[0.9592326657160306, 0.9608706187879862, 0.962..."
...,...,...,...,...,...,...,...
91,yolo8,Jellytoring,grey_world,NoGroups,0.915827,0.002817,"[0.919648629015166, 0.9148895229047502, 0.9129..."
92,yolo8,Jellytoring,ace,NoGroups,0.913742,0.002455,"[0.9135652545343936, 0.9168331088414957, 0.910..."
93,yolo8,Jellytoring,ARCR,NoGroups,0.912263,0.000378,"[0.9126948209220188, 0.9123207127338447, 0.911..."
94,yolo8,Jellytoring,MLLE,NoGroups,0.909043,0.001452,"[0.9074451707414273, 0.9109597790260754, 0.908..."


# MLLE combined with MSRCR

In [9]:
res_dir = 'MLLE_MSRCR_combi'
eval_type = 'Groups' # domain-based evaluation
dataset = 'DeepFish'
results_df = []
for model in ['yolo8', 'dt2']:
    for augm in [
        'MLLE_resized_MSRCR',
        'resized_MSRCR_MLLE'
    ]:
        train_data = f'{dataset}.CV.{eval_type}{f".{augm}" if augm else ""}'
        fn = os.path.join(results_dir, 'pickles', f'{res_dir}.{model}.{dataset}.{eval_type}.{augm}.pckl')
        if not os.path.exists(fn):
            results = utils.read_and_eval_results(
                train_datas=train_data,
                models=[model],
                model_selects=['best'],
                seeds=C.ALL_SEEDS[dataset],
                splits=['test'],
                results_dir=os.path.join(results_dir, res_dir),
                n_folds=N_FOLDS,
                concat=CCF,
                verbose=False,
            )
            utils.clean_cached_results(results)
            utils.pickle_dump(results, fn)
        else:
            results = utils.pickle_load(fn)

        r = results[train_data][model]['best']['test']
        aps = [r[s].precision.mean() for s in r]
        print(f'{model} {train_data} {np.mean(aps):.3f} {np.std(aps):.3f} {[round(ap, 3) for ap in aps]}')
        results_df.append((model, dataset, augm, eval_type, np.mean(aps), np.std(aps), aps))
results_df = pd.DataFrame.from_records(results_df, columns=['model', 'dataset', 'pps', 'eval', 'mAP', 'std', 'mAPs'])
results_df.to_csv(os.path.join(results_dir, f'{res_dir}.csv'), index=None)
results_df

yolo8 DeepFish.CV.Groups.MLLE_resized_MSRCR 0.486 0.026 [0.464, 0.523, 0.471]
yolo8 DeepFish.CV.Groups.resized_MSRCR_MLLE 0.537 0.062 [0.573, 0.588, 0.45]
dt2 DeepFish.CV.Groups.MLLE_resized_MSRCR 0.487 0.042 [0.445, 0.545, 0.472]
dt2 DeepFish.CV.Groups.resized_MSRCR_MLLE 0.529 0.022 [0.559, 0.519, 0.509]


,model,dataset,pps,eval,mAP,std,mAPs
0,yolo8,DeepFish,MLLE_resized_MSRCR,Groups,0.485876,0.026191,"[0.46421049413715937, 0.5227259785165099, 0.47..."
1,yolo8,DeepFish,resized_MSRCR_MLLE,Groups,0.537132,0.061568,"[0.5730054276468635, 0.5879019452692945, 0.450..."
2,dt2,DeepFish,MLLE_resized_MSRCR,Groups,0.487162,0.042317,"[0.44463703694020584, 0.5448917615573152, 0.47..."
3,dt2,DeepFish,resized_MSRCR_MLLE,Groups,0.528898,0.021822,"[0.559222268058264, 0.5186969427629908, 0.5087..."


# Training data subsampling

In [10]:
res_dir = 'SUBSAMPLE'
eval_type = 'Groups' # domain-based evaluation
results_df = []
dataset = 'DeepFish'
for model in ['yolo8', 'dt2']:
    for augm in [
        'ssT400',
        'ssT400.ssG50',
        'ssT400.ssG25',
        'ssT1000',
        'ssT1000.ssG50',
    ]:
        for replicate in ['x1', 'x2', 'x3']:
            train_data = f'{dataset}.CV.{eval_type}{f".{augm}.{replicate}" if augm else ""}'
            fn = os.path.join(results_dir, 'pickles', f'{res_dir}.{model}.{dataset}.{eval_type}.{augm}.{replicate}.pckl')
            if not os.path.exists(fn):
                results = utils.read_and_eval_results(
                    train_datas=train_data,
                    models=[model],
                    model_selects=['best'],
                    seeds=C.ALL_SEEDS[dataset],
                    splits=['test'],
                    results_dir=os.path.join(results_dir, res_dir),
                    n_folds=N_FOLDS,
                    concat=CCF,
                    verbose=False,
                )
                utils.clean_cached_results(results)
                utils.pickle_dump(results, fn)
            else:
                results = utils.pickle_load(fn)

            r = results[train_data][model]['best']['test']
            aps = [r[s].precision.mean() for s in r]
            print(f'{model} {train_data} {np.mean(aps):.3f} {np.std(aps):.3f} {[round(ap, 3) for ap in aps]}')
            results_df.append((model, dataset, augm, eval_type, np.mean(aps), np.std(aps), aps))
results_df = pd.DataFrame.from_records(results_df, columns=['model', 'dataset', 'pps', 'eval', 'mAP', 'std', 'mAPs'])
results_df.to_csv(os.path.join(results_dir, f'{res_dir}.csv'), index=None)
results_df

yolo8 DeepFish.CV.Groups.ssT400.x1 0.414 0.041 [0.357, 0.455, 0.428]
yolo8 DeepFish.CV.Groups.ssT400.x2 0.392 0.013 [0.396, 0.375, 0.406]
yolo8 DeepFish.CV.Groups.ssT400.x3 0.402 0.040 [0.391, 0.455, 0.36]
yolo8 DeepFish.CV.Groups.ssT400.ssG50.x1 0.328 0.056 [0.321, 0.4, 0.263]
yolo8 DeepFish.CV.Groups.ssT400.ssG50.x2 0.328 0.056 [0.321, 0.4, 0.263]
yolo8 DeepFish.CV.Groups.ssT400.ssG50.x3 0.328 0.056 [0.321, 0.4, 0.263]
yolo8 DeepFish.CV.Groups.ssT400.ssG25.x1 0.247 0.079 [0.242, 0.347, 0.154]
yolo8 DeepFish.CV.Groups.ssT400.ssG25.x2 0.261 0.124 [0.302, 0.388, 0.093]
yolo8 DeepFish.CV.Groups.ssT400.ssG25.x3 0.283 0.018 [0.302, 0.26, 0.287]
yolo8 DeepFish.CV.Groups.ssT1000.x1 0.419 0.053 [0.357, 0.487, 0.414]
yolo8 DeepFish.CV.Groups.ssT1000.x2 0.432 0.035 [0.385, 0.442, 0.469]
yolo8 DeepFish.CV.Groups.ssT1000.x3 0.435 0.068 [0.429, 0.521, 0.355]
yolo8 DeepFish.CV.Groups.ssT1000.ssG50.x1 0.386 0.031 [0.343, 0.399, 0.416]
yolo8 DeepFish.CV.Groups.ssT1000.ssG50.x2 0.390 0.053 [0.35, 0.46

,model,dataset,pps,eval,mAP,std,mAPs
0,yolo8,DeepFish,ssT400,Groups,0.413543,0.041272,"[0.35732015066263945, 0.45523214742739676, 0.4..."
1,yolo8,DeepFish,ssT400,Groups,0.391975,0.012835,"[0.39566839465810344, 0.374737860823817, 0.405..."
2,yolo8,DeepFish,ssT400,Groups,0.402255,0.039612,"[0.3910764335248845, 0.4553836006112906, 0.360..."
3,yolo8,DeepFish,ssT400.ssG50,Groups,0.328337,0.056239,"[0.3213758643183255, 0.4004316375321396, 0.263..."
4,yolo8,DeepFish,ssT400.ssG50,Groups,0.328337,0.056239,"[0.3213758643183255, 0.4004316375321396, 0.263..."
5,yolo8,DeepFish,ssT400.ssG50,Groups,0.328337,0.056239,"[0.3213758643183255, 0.4004316375321396, 0.263..."
6,yolo8,DeepFish,ssT400.ssG25,Groups,0.247436,0.079007,"[0.24150644827476633, 0.34702743396645913, 0.1..."
7,yolo8,DeepFish,ssT400.ssG25,Groups,0.261061,0.123806,"[0.30186434381770244, 0.38811500873929017, 0.0..."
8,yolo8,DeepFish,ssT400.ssG25,Groups,0.283219,0.017598,"[0.302341631285015, 0.25986290568369863, 0.287..."
9,yolo8,DeepFish,ssT1000,Groups,0.419275,0.053394,"[0.3568951349812978, 0.48731601761424476, 0.41..."


# Comparing different configurations of MSRCR

In [11]:
eval_type = 'Groups' # domain-based evaluation
dataset = 'DeepFish'
results_df = []
for model in [
    'yolo8',
]:
    for augm, res_dir in [
        ('resized_MSRCR_5_27_83_new', 'SIGMAS_NEW'),
        ('resized_MSRCR_8_40_125_new', 'SIGMAS_NEW'),
        ('resized_MSRCR_15_80_250_new', 'FINAL_GROUPS'),
        ('resized_MSRCR_30_160_500_new', 'SIGMAS_NEW'),

        ('resized_MSRCR_15_80_new', 'SIGMAS_NEW'),
        ('resized_MSRCR_80_250_new', 'SIGMAS_NEW'),
        ('resized_MSRCR_15_250_new', 'SIGMAS_NEW'),
        
        ('resized_SSR_15_new', 'SIGMAS_NEW'),
        ('resized_SSR_80_new', 'SIGMAS_NEW'),
        ('resized_SSR_250_new', 'SIGMAS_NEW'),
    ]:
        train_data = f'{dataset}.CV.{eval_type}{f".{augm}" if augm else ""}'
        fn = os.path.join(results_dir, 'pickles', f'{res_dir}.{model}.{dataset}.{eval_type}.{augm}.pckl')
        if not os.path.exists(fn):
            results = utils.read_and_eval_results(
                train_datas=train_data,
                models=[model],
                model_selects=['best'],
                seeds=C.ALL_SEEDS[dataset],
                splits=['test'],
                results_dir=os.path.join(results_dir, res_dir),
                n_folds=N_FOLDS,
                concat=CCF,
                verbose=False,
            )
            utils.clean_cached_results(results)
            utils.pickle_dump(results, fn)
        else:
            results = utils.pickle_load(fn)

        r = results[train_data][model]['best']['test']
        aps = [r[s].precision.mean() for s in r]
        print(f'{model} {train_data} {np.mean(aps):.3f} {np.std(aps):.3f} {[round(ap, 3) for ap in aps]}')
        results_df.append((model, dataset, augm, eval_type, np.mean(aps), np.std(aps), aps))
assert res_dir == 'SIGMAS_NEW'
results_df = pd.DataFrame.from_records(results_df, columns=['model', 'dataset', 'pps', 'eval', 'mAP', 'std', 'mAPs'])
results_df.to_csv(os.path.join(results_dir, f'{res_dir}.csv'), index=None)
results_df

yolo8 DeepFish.CV.Groups.resized_MSRCR_5_27_83_new 0.547 0.014 [0.556, 0.526, 0.558]
yolo8 DeepFish.CV.Groups.resized_MSRCR_8_40_125_new 0.527 0.016 [0.545, 0.529, 0.506]
yolo8 DeepFish.CV.Groups.resized_MSRCR_15_80_250_new 0.551 0.012 [0.566, 0.55, 0.535]
yolo8 DeepFish.CV.Groups.resized_MSRCR_30_160_500_new 0.549 0.007 [0.549, 0.54, 0.557]
yolo8 DeepFish.CV.Groups.resized_MSRCR_15_80_new 0.509 0.049 [0.451, 0.571, 0.504]
yolo8 DeepFish.CV.Groups.resized_MSRCR_80_250_new 0.506 0.035 [0.515, 0.46, 0.544]
yolo8 DeepFish.CV.Groups.resized_MSRCR_15_250_new 0.522 0.053 [0.47, 0.595, 0.5]
yolo8 DeepFish.CV.Groups.resized_SSR_15_new 0.519 0.027 [0.534, 0.542, 0.481]
yolo8 DeepFish.CV.Groups.resized_SSR_80_new 0.496 0.032 [0.453, 0.505, 0.53]
yolo8 DeepFish.CV.Groups.resized_SSR_250_new 0.501 0.044 [0.44, 0.537, 0.526]


,model,dataset,pps,eval,mAP,std,mAPs
0,yolo8,DeepFish,resized_MSRCR_5_27_83_new,Groups,0.546528,0.014341,"[0.5557503219124639, 0.5262731188938597, 0.557..."
1,yolo8,DeepFish,resized_MSRCR_8_40_125_new,Groups,0.526584,0.016120,"[0.5451763619049933, 0.5287111399127902, 0.505..."
2,yolo8,DeepFish,resized_MSRCR_15_80_250_new,Groups,0.550555,0.012407,"[0.565872777608837, 0.5503089366107838, 0.5354..."
3,yolo8,DeepFish,resized_MSRCR_30_160_500_new,Groups,0.548746,0.006926,"[0.5487436074688262, 0.5402641827776732, 0.557..."
4,yolo8,DeepFish,resized_MSRCR_15_80_new,Groups,0.508721,0.049134,"[0.4512142405555127, 0.5712528214416791, 0.503..."
5,yolo8,DeepFish,resized_MSRCR_80_250_new,Groups,0.506134,0.034523,"[0.5145202927897201, 0.4602870678144416, 0.543..."
6,yolo8,DeepFish,resized_MSRCR_15_250_new,Groups,0.521640,0.053237,"[0.46987427512774277, 0.5948677521571623, 0.50..."
7,yolo8,DeepFish,resized_SSR_15_new,Groups,0.519078,0.027006,"[0.5340685113681319, 0.5420032033622013, 0.481..."
8,yolo8,DeepFish,resized_SSR_80_new,Groups,0.496178,0.031982,"[0.4533599625602039, 0.5049684625226986, 0.530..."
9,yolo8,DeepFish,resized_SSR_250_new,Groups,0.500885,0.043582,"[0.43953469209438134, 0.5366868853670431, 0.52..."
